# Preprocessing and Exploratory Data Analysis
This notebook covers the initial preprocessing of the clinical dataset for the comorbidity inductive bias experiments. 
It cleans the raw dataset and explores the target labels (comorbidities).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configure styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')


## 1. Load Data
We load the data from the `../data/data.csv` location.

In [ ]:
data_path = '../data/data.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"Loaded raw dataset with {len(df)} records and {len(df.columns)} features.")
    display(df.head())
else:
    print(f"Data file not found at {data_path}. Please place the open-sourced dataset there.")


## 2. Clean 'DIA LIFE' Feature
The `DIA LIFE` column contains a mix of years and months. We convert all values to years.

In [ ]:
def clean_dia_life(x):
    if pd.isna(x):
        return np.nan
    x_str = str(x).lower().strip()
    if 'month' in x_str or x_str.endswith('m'):
        num = ''.join(filter(lambda c: c.isdigit() or c == '.', x_str))
        try:
            return float(num) / 12
        except:
            return np.nan
    else:
        try:
            return float(x_str)
        except:
            return np.nan

if 'df' in locals() and 'DIA LIFE' in df.columns:
    df['DIA LIFE'] = df['DIA LIFE'].apply(clean_dia_life)
    print("Cleaned DIA LIFE column.")


## 3. Handle Missing Values
We remove rows with missing features or targets to ensure a clean dataset for training.

In [ ]:
complications = ['NEP', 'NEU', 'RET']
if 'df' in locals():
    exclude_cols = ['SL.NO', 'NAME'] + complications
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    # Drop missing values
    df_clean = df.dropna(subset=feature_cols + complications)
    print(f"Records remaining after dropping missing values: {len(df_clean)}")


## 4. Exploratory Data Analysis (Comorbidities)
Since our study focuses on comorbidities, let's analyze the co-occurrence of the target complications.

In [ ]:
if 'df_clean' in locals():
    # Individual frequencies
    freq = df_clean[complications].mean() * 100
    
    plt.figure(figsize=(8, 5))
    bars = plt.bar(freq.index, freq.values, color=['#4C72B0', '#DD8452', '#55A868'])
    plt.title('Prevalence of Individual Complications (%)', fontsize=14)
    plt.ylabel('Prevalence (%)')
    plt.ylim(0, 100)
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f'{yval:.1f}%', ha='center', va='bottom', fontsize=11)
    
    plt.tight_layout()
    plt.show()


In [ ]:
if 'df_clean' in locals():
    # Co-occurrence matrix
    co_occ = df_clean[complications].T.dot(df_clean[complications])
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(co_occ, annot=True, fmt='g', cmap='Blues', cbar=False)
    plt.title('Comorbidity Co-occurrence Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()


## 5. Save Preprocessed Data
We save the cleaned dataset to the `notebooks/` directory so the experiment notebooks can easily load it.

In [ ]:
if 'df_clean' in locals():
    # The other notebooks expect 'data.csv' in the same directory
    df_clean.to_csv('data.csv', index=False)
    print("Saved preprocessed dataset to 'data.csv'.")
